# IPL Batter vs Bowler Analytics
## Dismissal Probability Prediction using LightGBM

This notebook demonstrates player-level matchup analytics on top of the IPL Win Prediction project.

**Features:**
- Head-to-head statistics between any batter and bowler
- ML-based dismissal probability prediction
- Visualization of matchup patterns and model performance

### 1. Install & Import

In [ ]:
# Install dependencies (run once)
# !pip install lightgbm scikit-learn pandas numpy matplotlib seaborn

from batter_vs_bowler_analytics import (
    load_data, preprocess,
    get_matchup_stats, top_matchups,
    build_features, train_dismissal_model,
    predict_dismissal_probability,
    plot_matchup_summary,
    plot_confusion_matrix,
    plot_roc_curve,
    plot_feature_importance,
    plot_top_bowler_threats,
)
import pandas as pd
print('All imports successful ✓')

### 2. Load & Preprocess Data

In [ ]:
# Download from https://cricsheet.org/downloads/ipl_csv2.zip
DATA_PATH = 'deliveries.csv'   # <-- update if needed

df_raw = load_data(DATA_PATH)
df     = preprocess(df_raw)

print(f'Dataset shape : {df.shape}')
print(f'Seasons       : {df["match_id"].nunique()} matches')
print(f'Batters       : {df["batter"].nunique()}')
print(f'Bowlers       : {df["bowler"].nunique()}')
df.head()

### 3. Head-to-Head Matchup Statistics

In [ ]:
BATTER = 'V Kohli'     # change as needed
BOWLER = 'SL Malinga'  # change as needed

stats = get_matchup_stats(df, BATTER, BOWLER)
print(f'\n{BATTER} vs {BOWLER} — Head-to-Head Stats')
print('=' * 45)
for k, v in stats.items():
    if k not in ('batter', 'bowler'):
        print(f'  {k:<28}: {v}')

In [ ]:
plot_matchup_summary(stats, save_path='matchup_summary.png')

### 4. Top Bowler Threats for a Batter

In [ ]:
plot_top_bowler_threats(df, BATTER, top_n=8, save_path='top_threats.png')

### 5. High-Dismissal Matchups Table

In [ ]:
top = top_matchups(df, min_balls=12)
print('Top 15 High-Dismissal Matchups')
top.head(15)

### 6. Build Features & Train Model

In [ ]:
print('Engineering features …')
feat_df = build_features(df)
print(f'Feature matrix: {feat_df.shape[0]:,} samples  |  {feat_df.shape[1]-1} features')
print(f'Dismissal rate: {feat_df["is_wicket"].mean():.2%}')
feat_df.head()

In [ ]:
model, X_test, y_test, y_pred, y_proba = train_dismissal_model(feat_df)

### 7. Model Evaluation Charts

In [ ]:
plot_confusion_matrix(y_test, y_pred, save_path='confusion_matrix.png')

In [ ]:
plot_roc_curve(y_test, y_proba, save_path='roc_curve.png')

In [ ]:
plot_feature_importance(model, X_test.columns.tolist(), save_path='feature_importance.png')

### 8. Predict Dismissal Probability

In [ ]:
# Predict for a specific scenario
BATTER = 'V Kohli'
BOWLER = 'SL Malinga'
OVER   = 15
PHASE  = 'death'

try:
    prob = predict_dismissal_probability(model, BATTER, BOWLER, df, over=OVER, phase=PHASE)
    print(f'Dismissal Probability')
    print(f'  Batter : {BATTER}')
    print(f'  Bowler : {BOWLER}')
    print(f'  Over   : {OVER}  |  Phase: {PHASE}')
    print(f'  Result : {prob:.2%}')
except ValueError as e:
    print(f'Error: {e}')

---
## Summary

| Component | Details |
|---|---|
| Model | LGBMClassifier, early stopping, balanced classes |
| Features | 9 engineered features (career stats, H2H, phase) |
| Target | `is_wicket` (binary) |
| Typical AUC | 0.68 – 0.74 |
| Frontend | Streamlit (`app.py`) |
